# Redefinición del conjunto de datos del TFG: evitando el data leakage

In [15]:
import numpy as np
import pandas as pd
from itertools import permutations, product
from random import seed, shuffle, random
import math
import time

## Nuevos conjuntos de datos

Para crear los nuevos conjuntos de datos, uno para cada fase del modelo (train, validation y test), se seguirá la siguiente estrategia:

- Permutaciones sin repetición. En particular: 4 datasets para train, 3 para validation, 3 para test
  
- Barajado de permutaciones
  
En cuanto al tamaño de los conjuntos de datos, se tiene, por un lado:

- 4 datasets train -> 4! = 24 permutaciones

- 3 datasets validation -> 3! = 6 permutaciones

- 3 datasets test -> 3! = 6 permutaciones

Por otro lado, cada dataset se transforma en una serie temporal con 30 intervalos. Por lo tanto:

- Train: 24 permutaciones x 4 datasets/permutación x 30 intervalos/dataset = 2880 intervalos

- Validation: 6 permutaciones x 3 datasets/permutación x 30 intervalos/dataset = 540 puntos

- Test: 6 permutaciones x 3 datasets/permutación x 30 intervalos/dataset = 540 puntos

### Construcción de las permutaciones

In [3]:
#############################################
################## TRAIN ####################
#############################################

# Establecer semilla
seed(10)

# Lista de números
nums_train = [0, 3, 5, 7]

# Generar todas las permutaciones
perm_train = list(permutations(nums_train))

# Mezclar las permutaciones
shuffle(perm_train)

In [4]:
perm_train

[(5, 7, 0, 3),
 (0, 5, 7, 3),
 (3, 5, 7, 0),
 (5, 7, 3, 0),
 (3, 7, 5, 0),
 (7, 5, 0, 3),
 (5, 0, 3, 7),
 (0, 7, 5, 3),
 (3, 0, 7, 5),
 (3, 5, 0, 7),
 (7, 0, 5, 3),
 (0, 5, 3, 7),
 (3, 7, 0, 5),
 (0, 7, 3, 5),
 (7, 3, 5, 0),
 (7, 3, 0, 5),
 (5, 3, 0, 7),
 (3, 0, 5, 7),
 (0, 3, 5, 7),
 (7, 5, 3, 0),
 (5, 3, 7, 0),
 (5, 0, 7, 3),
 (0, 3, 7, 5),
 (7, 0, 3, 5)]

In [5]:
#############################################
############### VALIDATION ##################
#############################################

# Establecer semilla
seed(68)

# Lista de números
nums_validation = [1, 6, 9]

# Generar todas las permutaciones
perm_validation = list(permutations(nums_validation))

# Mezclar las permutaciones
shuffle(perm_validation)

In [6]:
perm_validation

[(1, 9, 6), (9, 1, 6), (6, 1, 9), (1, 6, 9), (6, 9, 1), (9, 6, 1)]

In [7]:
#############################################
################### TEST ####################
#############################################

# Establecer la semilla
seed(42)

# Paso 1: Generar permutaciones de los datasets
nums_test = [2, 4, 8]
perm_test = list(permutations(nums_test))
shuffle(perm_test)

In [8]:
perm_test

[(4, 8, 2), (2, 8, 4), (4, 2, 8), (8, 2, 4), (2, 4, 8), (8, 4, 2)]

In [9]:
len(perm_train)

24

In [10]:
len(perm_validation)

6

In [11]:
len(perm_test)

6

### Funciones para la creación del con

In [18]:
data_path = "../data"

In [14]:
def add_dataset_ID_column(df, i):
    df['dataset_ID'] = i
    return df
    
def add_interval_column(df, long_interval, i):
    global EXECUTION
    bins = range(0, 3000 + long_interval, long_interval)  # 0 a 3000 ms en intervalos de long_interval ms
    # Agregamos una nueva columna al DataFrame con las etiquetas de los intervalos
    df.loc[:, 'interval'] = EXECUTION * 30 + (pd.cut(df['timestamps'], bins=bins, labels=False) + 1)
    return df

def delete_unnecessary_columns(df):
    df_nuevo = df[["timestamps"]].copy()
    return df_nuevo

def add_spikes_column(df):
    df = df.groupby(['dataset_ID', 'interval']).size().reset_index(name='spikes')
    return df

def lectura_datasets(long_interval, lista, attack):
    global EXECUTION
    
    if attack == "spontaneous":
        # Ruta base del archivo CSV para caso espontáneo
        base_path = data_path + "csv/output_flash_trial_{}_BKG_trial_9{}/spikes.csv"
    else:
        # Ruta base del archivo CSV para el caso de ataque FLO
        base_path = data_path + "csv/output_FLO_" + attack + "_flash_trial_{}_BKG_trial_9{}/spikes.csv"
            
    # DataFrame vacío para almacenar todos los datos
    combined_dataset = pd.DataFrame()
    
    for i in lista:
        # Genera la ruta completa del archivo CSV para el valor actual de i
        file_path = base_path.format(i, i)
        
        try:
            # Lee el archivo CSV
            df = pd.read_csv(file_path, delimiter=";")
            df = delete_unnecessary_columns(df)
            df = add_dataset_ID_column(df, i)
            df = add_interval_column(df, long_interval, i)
            df = add_spikes_column(df)
            # Concatena el dataframe al conjunto de datos consolidado
            combined_dataset = pd.concat([combined_dataset, df], ignore_index=True)
            EXECUTION = EXECUTION + 1
        
        except FileNotFoundError:
            print(f"Archivo no encontrado para i = {i}")

    return combined_dataset

In [17]:
start_time = time.time()

# Creación del conjunto de train usando simulaciones espontáneas
train_df = pd.DataFrame()
for perm in perm_train:
    train_df = pd.concat([train_df, lectura_datasets(INTERVAL_LENGTH, perm, "spontaneous")], ignore_index=True)

train_df.to_csv(data_path + "train_Flash_Semisupervisado.csv", index = False)

# Creación del conjunto de validation usando simulaciones espontáneas
validation_df = pd.DataFrame()
for perm in perm_validation:
    validation_df = pd.concat([validation_df, lectura_datasets(INTERVAL_LENGTH, perm, "spontaneous")], ignore_index=True)
validation_df.to_csv(data_path + "validation_Flash_Semisupervisado.csv", index = False)

# Creación del conjunto de test usando simulaciones de ataque
test_df = pd.DataFrame()
for perm in perm_test:
    test_df = pd.concat([test_df, lectura_datasets(INTERVAL_LENGTH, perm, "quarter")], ignore_index=True)

test_df.to_csv(data_path + "test_Flash_Semisupervisado.csv", index = False)

elapsed_time = time.time() - start_time
display(f"Tiempo transcurrido: {elapsed_time} segundos")

'Tiempo transcurrido: 67.18659448623657 segundos'